# TermNorm Backend

Connect to TermNorm, sync experiments, replay pipelines, compare variants.

**Prerequisites:** TermNorm running at `http://127.0.0.1:8000`

In [ ]:
#@title Setup & imports
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from api.models.backend import BackendConnection
from api.services.project_store import ProjectStore
from api.services.backend_client import BackendClient

TERMNORM_URL = "http://127.0.0.1:8000"
BACKEND_ID = "termnorm-local"

store = ProjectStore(base_dir=PROJECT_ROOT / ".promptpotter" / "projects")
client = BackendClient(TERMNORM_URL)

print(f"Project root: {PROJECT_ROOT}")
print(f"Store: {store.base_dir}")
print("Ready")

## 1. Register backend

In [ ]:
#@title Register backend (idempotent)
if store.get_backend(BACKEND_ID):
    backend = store.get_backend(BACKEND_ID)
    print(f"Already registered: {backend.name} ({backend.base_url})")
else:
    backend = BackendConnection(
        id=BACKEND_ID,
        name="TermNorm Local",
        backend_type="termnorm",
        base_url=TERMNORM_URL,
    )
    store.register_backend(backend)
    print(f"Registered: {backend.name}")

print(f"Store: .promptpotter/projects/{BACKEND_ID}/")

## 2. Sync experiments from TermNorm

In [ ]:
#@title Sync experiments from TermNorm
count = await client.sync_experiments(store, BACKEND_ID)

# Update last_synced_at
from datetime import datetime, timezone
backend.last_synced_at = datetime.now(timezone.utc).isoformat()
store.update_backend(backend)

print(f"Synced {count} experiment(s)")

In [ ]:
#@title List synced experiments
experiments = store.load_sync(BACKEND_ID, "experiments.json")
for exp in experiments.get("experiments", []):
    exp_id = exp.get("experiment_id", exp.get("id", "?"))
    print(f"  {exp_id}: {exp.get('name', '')} — {exp.get('description', '')[:80]}")

## 3. Inspect a synced experiment

In [ ]:
#@title Extract replay queries & session terms
EXPERIMENT_ID = "1_production_historical"  # adjust if needed

exp_data = store.load_sync(BACKEND_ID, f"experiments/{EXPERIMENT_ID}.json")

terms = client.extract_session_terms(exp_data)
queries = client.extract_replay_queries(exp_data)

print(f"Experiment: {exp_data.get('experiment', {}).get('name', EXPERIMENT_ID)}")
print(f"Mappings: {len(exp_data.get('mappings', []))}")
print(f"Session terms: {len(terms)}")
print(f"Replayable queries (with ground truth): {len(queries)}")
print()
print("First 5 queries:")
for q in queries[:5]:
    print(f"  {q['query'][:60]}  ->  GT: {q['ground_truth'][:50]}")

In [ ]:
#@title Experiment overview
exp_meta = exp_data.get("experiment", {})
runs = exp_data.get("runs", [])

print("EXPERIMENT OVERVIEW")
print("=" * 60)
print(f"  ID:              {exp_meta.get('experiment_id', '?')}")
print(f"  Name:            {exp_meta.get('name', '?')}")
print(f"  Description:     {exp_meta.get('description', '?')}")
print(f"  Lifecycle stage: {exp_meta.get('lifecycle_stage', '?')}")
print(f"  Mappings count:  {exp_data.get('mappings_count', len(exp_data.get('mappings', [])))}")
print(f"  Runs:            {len(runs)}")

if exp_meta.get("creation_time"):
    from datetime import datetime
    created = datetime.fromtimestamp(exp_meta["creation_time"] / 1000)
    print(f"  Created:         {created.isoformat()}")

for i, r in enumerate(runs):
    print(f"\n  Run {i}: {r.get('run_name', r.get('run_id', '?'))}")
    print(f"    ID:     {r.get('run_id', '?')}")
    print(f"    Status: {r.get('status', '?')}")
    print(f"    Tags:   {r.get('tags', {})}")

In [ ]:
#@title Run parameters & pipeline steps
import pandas as pd

run = exp_data["runs"][0]

# Run parameters
print("RUN PARAMETERS")
print("-" * 40)
params_df = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in run["params"].items()]
)
display(params_df)

# Pipeline config
pipeline = run.get("pipeline", {})
config = pipeline.get("config", {})
steps = config.get("steps", [])

print(f"\nPIPELINE: {config.get('name', '?')} ({config.get('version', '?')})")
print(f"Description: {config.get('description', '')}")
print(f"Notation: {pipeline.get('notation', '?')}")
print()

# Pipeline steps table
step_rows = []
for s in steps:
    step_rows.append({
        "step": s["name"],
        "type": s["type"],
        "inputs": ", ".join(s.get("signature", {}).get("input_fields", [])),
        "outputs": ", ".join(s.get("signature", {}).get("output_fields", [])),
        "model": s.get("config", {}).get("model", "—"),
        "temperature": s.get("config", {}).get("temperature", "—"),
    })
steps_df = pd.DataFrame(step_rows)
display(steps_df)

In [ ]:
#@title Run metrics (baseline — full pipeline with LLM2)
metrics = run["metrics"]

print("RUN METRICS (baseline — full pipeline with LLM2)")
print("=" * 50)
print(f"  Queries evaluated:  {int(metrics.get('num_queries', 0))}")
print(f"  hit@1 (Accuracy):   {metrics.get('hit_at_1', 0):.1%}")
print(f"  hit@3:              {metrics.get('hit_at_3', 0):.1%}")
print(f"  hit@5:              {metrics.get('hit_at_5', 0):.1%}")
print(f"  MRR:                {metrics.get('mrr', 0):.3f}")
print(f"  Avg latency:        {metrics.get('avg_latency_ms', 0):,.0f} ms")
print(f"  Avg confidence:     {metrics.get('avg_confidence', 0):.3f}")

In [ ]:
#@title Evaluation results from stored run
eval_results = run.get("evaluation_results", [])
eval_df = pd.DataFrame(eval_results)

print(f"EVALUATION RESULTS: {len(eval_df)} queries")
print("=" * 50)

if not eval_df.empty:
    print(f"\nLatency (ms):")
    print(f"  Mean:     {eval_df['latency_ms'].mean():,.0f}")
    print(f"  Median:   {eval_df['latency_ms'].median():,.0f}")
    print(f"  Min:      {eval_df['latency_ms'].min():,.0f}")
    print(f"  Max:      {eval_df['latency_ms'].max():,.0f}")

    print(f"\nConfidence:")
    print(f"  Mean:     {eval_df['confidence'].mean():.3f}")
    print(f"  Non-zero: {(eval_df['confidence'] > 0).sum()} / {len(eval_df)}")

    print(f"\nMethods: {eval_df['method'].value_counts().to_dict()}")

    print(f"\nFirst 10 results:")
    display(eval_df[["query", "predicted", "method", "confidence", "latency_ms"]].head(10))

In [ ]:
#@title Mappings overview
mappings = exp_data.get("mappings", [])
total = len(mappings)
with_gt = sum(1 for m in mappings if m.get("dataset_entry", "").strip() not in ("", "--"))
no_gt = total - with_gt

print(f"MAPPINGS OVERVIEW")
print("=" * 50)
print(f"  Total mappings:          {total}")
print(f"  With ground truth:       {with_gt}")
print(f"  Without ground truth:    {no_gt}  (empty or '--')")
if total:
    print(f"  Evaluation coverage:     {with_gt/total:.1%}")

# Sample mappings with ground truth
print(f"\nSample mappings (first 10 with ground truth):")
sample_rows = []
for m in mappings:
    if m.get("dataset_entry", "").strip() not in ("", "--"):
        sample_rows.append({
            "bom_material": m["bom_material"][:50],
            "dataset_entry": m["dataset_entry"][:60],
        })
    if len(sample_rows) >= 10:
        break
display(pd.DataFrame(sample_rows))

## 4. Replay (execute via TermNorm API)

Calls TermNorm's `/sessions` and `/matches` endpoints with `skip_llm_ranking=True`.

In [ ]:
#@title Replay pipeline (uses cache if available)
import uuid
from tqdm.auto import tqdm
from api.models.backend import Execution, ExecutionResultItem

VARIANT_LABEL = "LLM1-TokenMatching (no LLM2)"
PIPELINE_NOTATION = "LLM1-TokenMatching"
LIMIT = 0  # set to 0 for all queries, or a smaller number for quick tests

# Pipeline parameter overrides forwarded to TermNorm's /matches endpoint.
# Uncomment any key to override its default value.
PIPELINE_PARAMS = {
    # "max_sites": 7,              # Web scraping depth (pages fetched)
    # "num_results": 20,           # Search result count (Brave/SearXNG)
    # "content_char_limit": 800,   # Chars per scraped page
    # "raw_content_limit": 5000,   # Total research text to LLM1
    # "profiling_temperature": 0.3, # LLM1 entity profiling temperature
    # "profiling_max_tokens": 1800, # LLM1 output limit
    # "ranking_temperature": 0,    # LLM2 ranking temperature
    # "ranking_max_tokens": 4000,  # LLM2 output limit
    # "ranking_sample_size": 20,   # Candidates sent to LLM2
    # "max_token_candidates": 20,  # Candidates from token matching
    # "relevance_weight_core": 0.7, # Core concept weight in score
}

replay_queries_list = queries[:LIMIT] if LIMIT else queries
total = len(replay_queries_list)

# --- Check for existing execution with matching parameters ---
# Skip cache when pipeline_params are non-empty (params change results)
_cached = None
if not PIPELINE_PARAMS:
    for _ex in store.list_executions(BACKEND_ID):
        if (_ex["experiment_id"] == EXPERIMENT_ID and
            _ex["variant_label"] == VARIANT_LABEL and
            _ex["pipeline_notation"] == PIPELINE_NOTATION):
            _cached = store.load_execution(BACKEND_ID, _ex["execution_id"])
            if _cached:
                break

if _cached:
    # Reuse cached execution
    execution_id = _cached.execution_id
    execution = _cached
    results = [r.model_dump() for r in _cached.results]
    _hits = sum(1 for r in results if r.get("predicted") == r["ground_truth"])

    print(f"Using cached execution {execution_id}")
    print(f"  Queries: {len(results)}")
    print(f"  Accuracy (hit@1): {_hits}/{len(results)} ({_hits/len(results)*100:.1f}%)")
    print(f"  Created: {_cached.created_at}")
    print()
    print("(Delete the .json from executions/ to force a fresh replay)")
else:
    # No cache — run replay against TermNorm API
    execution_id = uuid.uuid4().hex[:12]

    if PIPELINE_PARAMS:
        print(f"Pipeline overrides: {PIPELINE_PARAMS}")
    print(f"Replaying {total} queries against {TERMNORM_URL}...")
    print(f"Execution ID: {execution_id}")
    print()

    # Progress tracking state
    _hits = 0
    _pbar = tqdm(total=total, desc="Replay", unit="query")

    async def on_result(result, index, total):
        """Save result incrementally and display rich feedback."""
        global _hits
        store.append_result(BACKEND_ID, execution_id, result)

        pred = result.get("predicted", "?")
        gt = result["ground_truth"]
        q = result["query"]
        latency = result.get("latency_ms", 0)
        conf = result.get("confidence", 0)
        hit = pred == gt

        if hit:
            _hits += 1

        tag = "HIT " if hit else "MISS"
        done = index + 1
        acc = _hits / done * 100

        tqdm.write(
            f"[{done}/{total}] {tag}  {q[:50]:<50s} "
            f"| pred: {pred[:35]:<35s} | GT: {gt[:35]:<35s} "
            f"| {latency:,.0f}ms | conf: {conf:.2f} "
            f"| Running: {_hits}/{done} ({acc:.1f}%)"
        )
        _pbar.update(1)

    results = await client.replay_queries(
        queries=replay_queries_list,
        terms=terms,
        skip_llm_ranking=True,
        delay_between=2.0,
        on_result=on_result,
        pipeline_params=PIPELINE_PARAMS,
    )
    _pbar.close()

    # Finalize: merge .jsonl into proper Execution .json
    successful = sum(1 for r in results if r["status"] == "success")
    errors = sum(1 for r in results if r["status"] == "error")
    total_latency = sum(r.get("latency_ms", 0) for r in results)
    total_conf = sum(r.get("confidence", 0) for r in results)

    execution = Execution(
        execution_id=execution_id,
        backend_id=BACKEND_ID,
        experiment_id=EXPERIMENT_ID,
        variant_label=VARIANT_LABEL,
        pipeline_notation=PIPELINE_NOTATION,
        session_terms_count=len(terms),
        pipeline_params=PIPELINE_PARAMS,
        query_count=len(results),
        successful_count=successful,
        error_count=errors,
        results=[ExecutionResultItem(**r) for r in results],
    )
    store.finalize_execution(execution)

    # End summary
    print()
    print("=" * 60)
    print(f"  Accuracy (hit@1): {_hits}/{total} ({_hits/total*100:.1f}%)")
    print(f"  Avg latency:      {total_latency/total:,.0f} ms")
    print(f"  Avg confidence:   {total_conf/total:.3f}")
    print(f"  Errors:           {errors}")
    print(f"  Execution ID:     {execution_id}")
    if PIPELINE_PARAMS:
        print(f"  Pipeline params:  {PIPELINE_PARAMS}")
    print("=" * 60)

In [ ]:
#@title Results table
import pandas as pd

rows = []
for r in results:
    gt = r["ground_truth"]
    pred = r.get("predicted", "")
    rows.append({
        "query": r["query"][:50],
        "predicted": pred[:50],
        "ground_truth": gt[:50],
        "correct": pred == gt,
        "latency_ms": r.get("latency_ms", 0),
    })

df = pd.DataFrame(rows)
display(df)

## 5. Compare variants

In [ ]:
from api.services.comparison import compute_comparison
import json

comparison = compute_comparison(
    results,
    metadata={
        "pipeline_notation": execution.pipeline_notation,
        "variant_label": execution.variant_label,
        "session_terms_count": execution.session_terms_count,
    },
)

m = comparison["metrics"]
c = comparison["classification"]
n = comparison["dataset"]["query_count"]

print(f"Queries: {n}")
print(f"")
print(f"Accuracy (hit@1):")
print(f"  Variant A (no LLM2): {m['hit_at_1']['a_count']}/{n} ({m['hit_at_1']['a']:.1%})")
print(f"  Variant B (full):    {m['hit_at_1']['b_count']}/{n} ({m['hit_at_1']['b']:.1%})")
print(f"")
print(f"Classification:")
print(f"  Both correct:   {c['both_correct']}")
print(f"  A-only correct: {c['a_only_correct']}  (LLM2 hurt)")
print(f"  B-only correct: {c['b_only_correct']}  (LLM2 helped)")
print(f"  Both wrong:     {c['both_wrong']}")

## 6. Browse stored executions

In [ ]:
#@title Browse stored executions
executions = store.list_executions(BACKEND_ID)
for ex in executions:
    print(f"  {ex['execution_id']}  {ex['variant_label']}  "
          f"queries={ex['query_count']}  success={ex['successful_count']}  "
          f"{ex['created_at']}")

## 7. Flat Search Prompt Optimization

Optimize the `llm_ranking` (reranker) prompt locally using captured `entity_profile` data.
Instead of full TermNorm replays (~40 min per candidate), we evaluate prompts locally (~5 min total).

In [ ]:
#@title Load baseline reranker prompt
exp_data = store.load_sync(BACKEND_ID, f"experiments/{EXPERIMENT_ID}.json")
dependencies = exp_data.get("dependencies", {})
prompts = dependencies.get("prompts", {})

# Find the llm_ranking prompt
reranker_prompt = None
for key, prompt_info in prompts.items():
    if "llm_ranking" in key:
        reranker_prompt = prompt_info
        break

if reranker_prompt is None:
    raise RuntimeError(
        "No llm_ranking prompt found in synced experiment data. "
        "Re-sync the experiment after TermNorm prompt registry is initialized."
    )

from api.models.prompt_state import PromptState

baseline = PromptState(
    instruction=reranker_prompt["template"],
    parameters={
        "family": reranker_prompt.get("family", "llm_ranking"),
        "version": reranker_prompt.get("version"),
        "template_variables": reranker_prompt.get("template_variables", []),
    },
    changes_description="Baseline reranker_v1 from TermNorm prompt registry",
)

print(f"Baseline prompt loaded: {baseline.id[:12]}")
print(f"  Family: {baseline.parameters['family']}")
print(f"  Version: {baseline.parameters['version']}")
print(f"  Template length: {len(baseline.instruction)} chars")
print(f"  Variables: {baseline.parameters['template_variables']}")

In [ ]:
#@title Load evaluation data from latest execution
import json

executions = store.list_executions(BACKEND_ID)

# Find the latest execution that has pipeline_data with entity_profile
eval_execution = None
for ex in sorted(executions, key=lambda e: e.get("created_at", ""), reverse=True):
    full = store.load_execution(BACKEND_ID, ex["execution_id"])
    if full is None:
        continue
    results = full.get("results", [])
    # Check if at least one result has entity_profile in pipeline_data
    has_profile = any(
        r.get("pipeline_data", {}).get("entity_profile") for r in results
    )
    if has_profile:
        eval_execution = full
        break

if eval_execution is None:
    print("WARNING: No execution found with pipeline_data.entity_profile.")
    print("Re-run Variant A replay (section 4) after TermNorm is updated to include entity_profile.")
    eval_data = []
else:
    eval_data = [
        r for r in eval_execution["results"]
        if r.get("status") == "success" and r.get("pipeline_data", {}).get("entity_profile")
    ]
    print(f"Loaded execution: {eval_execution['execution_id']}")
    print(f"  Variant: {eval_execution.get('variant_label', '?')}")
    print(f"  Queries with entity_profile: {len(eval_data)}/{len(eval_execution['results'])}")
    if eval_data:
        sample = eval_data[0]
        profile = sample["pipeline_data"]["entity_profile"]
        print(f"  Sample profile keys: {list(profile.keys())[:8]}")

In [ ]:
#@title Define local_reranker_eval()
import os
import random
import httpx

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
GROQ_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"


async def local_reranker_eval(prompt_template: str, query_data: dict) -> dict:
    """Evaluate a reranker prompt on a single query using cached pipeline data.

    Args:
        prompt_template: Prompt with {{core_concept}}, {{entity_profile_json}}, {{matches}} placeholders.
        query_data: A result dict with pipeline_data.entity_profile and pipeline_data.token_matched_candidates.

    Returns:
        dict with keys: query, predicted, ground_truth, hit, confidence, error
    """
    pipeline = query_data["pipeline_data"]
    entity_profile = pipeline["entity_profile"]
    candidates = pipeline.get("token_matched_candidates", [])
    ground_truth = query_data["ground_truth"]
    query = query_data["query"]

    # Build template variables (same as TermNorm's call_llm_for_ranking)
    core_concept = entity_profile.get("core_concept", "")
    entity_profile_json = json.dumps(entity_profile, indent=2)

    available = list(candidates[:20])
    sample_size = min(len(available), 20)
    sampled = random.sample(available, sample_size) if available else []
    matches = "\n".join(f"- {term}" for term, score in sampled)

    # Render prompt
    rendered = prompt_template.replace("{{core_concept}}", str(core_concept))
    rendered = rendered.replace("{{entity_profile_json}}", entity_profile_json)
    rendered = rendered.replace("{{matches}}", matches)

    # Add JSON instruction suffix (same as TermNorm)
    full_prompt = f"""{rendered}

IMPORTANT: Return a valid JSON response matching this exact structure:
{{
  "profile_summary": "Brief 1-2 sentence summary of the profile",
  "core_concept_description": "What the core concept fundamentally is",
  "ranked_candidates": [
    {{
      "candidate": "exact candidate string",
      "core_concept_score": 0.0,
      "spec_score": 0.0,
      "evaluation_reasoning": "Brief explanation without quotes or backslashes",
      "key_match_factors": ["factor1", "factor2"],
      "spec_gaps": ["gap1", "gap2"]
    }}
  ]
}}

Ensure all strings are properly escaped and avoid complex punctuation in reasoning."""

    try:
        async with httpx.AsyncClient() as http:
            resp = await http.post(
                GROQ_URL,
                headers={
                    "Authorization": f"Bearer {GROQ_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": GROQ_MODEL,
                    "messages": [{"role": "user", "content": full_prompt}],
                    "temperature": 0,
                    "max_tokens": 4000,
                    "response_format": {"type": "json_object"},
                },
                timeout=60.0,
            )
            resp.raise_for_status()
            llm_output = resp.json()["choices"][0]["message"]["content"]
            parsed = json.loads(llm_output)

        ranked = parsed.get("ranked_candidates", [])
        top = ranked[0] if ranked else {}
        predicted = top.get("candidate", "NO_RESULT")
        confidence = top.get("relevance_score", top.get("core_concept_score", 0))

        return {
            "query": query,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "hit": predicted == ground_truth,
            "confidence": confidence,
            "error": None,
        }
    except Exception as e:
        return {
            "query": query,
            "predicted": "ERROR",
            "ground_truth": ground_truth,
            "hit": False,
            "confidence": 0,
            "error": str(e),
        }


print("local_reranker_eval() defined")
print(f"  Model: {GROQ_MODEL}")
print(f"  API key set: {bool(GROQ_API_KEY)}")

In [ ]:
#@title Evaluate baseline prompt
import asyncio
from tqdm.auto import tqdm

assert eval_data, "No evaluation data — run the 'Load evaluation data' cell first"

baseline_results = []
_pbar = tqdm(total=len(eval_data), desc="Baseline eval", unit="query")

for qd in eval_data:
    result = await local_reranker_eval(baseline.render(), qd)
    baseline_results.append(result)

    tag = "HIT " if result["hit"] else "MISS"
    hits_so_far = sum(1 for r in baseline_results if r["hit"])
    done = len(baseline_results)
    acc = hits_so_far / done * 100

    tqdm.write(
        f"[{done}/{len(eval_data)}] {tag}  {result['query'][:50]:<50s} "
        f"| pred: {result['predicted'][:35]:<35s} "
        f"| GT: {result['ground_truth'][:35]:<35s} "
        f"| Running: {hits_so_far}/{done} ({acc:.1f}%)"
    )
    _pbar.update(1)

_pbar.close()

baseline_hits = sum(1 for r in baseline_results if r["hit"])
baseline_errors = sum(1 for r in baseline_results if r["error"])
baseline_accuracy = baseline_hits / len(baseline_results) if baseline_results else 0
avg_conf = sum(r["confidence"] for r in baseline_results) / len(baseline_results) if baseline_results else 0

print()
print("=" * 60)
print(f"BASELINE EVALUATION ({baseline.id[:12]})")
print(f"  Accuracy (hit@1): {baseline_hits}/{len(baseline_results)} ({baseline_accuracy:.1%})")
print(f"  Avg confidence:   {avg_conf:.3f}")
print(f"  Errors:           {baseline_errors}")
print("=" * 60)

In [ ]:
#@title Generate candidate prompts (flat search)
NUM_CANDIDATES = 5

# Analyze baseline failure patterns for the meta-prompt
failures = [r for r in baseline_results if not r["hit"] and not r["error"]]
failure_examples = "\n".join(
    f"  Query: {r['query'][:60]}  |  Predicted: {r['predicted'][:40]}  |  GT: {r['ground_truth'][:40]}"
    for r in failures[:10]
)

baseline_rendered = baseline.render()
meta_prompt = f"""You are a prompt engineering expert. Your task is to generate {NUM_CANDIDATES} improved variants
of a candidate-ranking prompt used in a terminology normalization pipeline.

CURRENT PROMPT (baseline — {baseline_accuracy:.1%} accuracy on {len(baseline_results)} queries):
---
{baseline_rendered}
---

FAILURE EXAMPLES (predicted != ground_truth):
{failure_examples}

The prompt takes these template variables (double-brace syntax):
  {{{{core_concept}}}} — the core concept word from the entity profile
  {{{{entity_profile_json}}}} — full JSON entity profile from web research
  {{{{matches}}}} — newline-separated list of "- candidate_term" from token matching

Generate exactly {NUM_CANDIDATES} variant prompts. For each variant:
1. Analyze WHY the baseline fails on the examples above
2. Make targeted changes to improve ranking accuracy (focus on getting the correct candidate to rank #1)
3. Keep the same template variables ({{{{core_concept}}}}, {{{{entity_profile_json}}}}, {{{{matches}}}})
4. Keep the same output format expectations (JSON with ranked_candidates)

Return a JSON array of objects, each with:
  - "variant_name": short identifier (e.g. "exact_match_focus")
  - "changes_description": 1-2 sentence description of what changed
  - "prompt_text": the full prompt template text

Return ONLY the JSON array, no other text."""

print(f"Generating {NUM_CANDIDATES} candidate prompts...")
print(f"  Baseline accuracy: {baseline_accuracy:.1%}")
print(f"  Failure examples provided: {len(failures[:10])}")

async with httpx.AsyncClient() as http:
    resp = await http.post(
        GROQ_URL,
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": GROQ_MODEL,
            "messages": [{"role": "user", "content": meta_prompt}],
            "temperature": 0.7,
            "max_tokens": 16000,
            "response_format": {"type": "json_object"},
        },
        timeout=120.0,
    )
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"]
    generated = json.loads(raw)

# Handle both {"variants": [...]} and bare [...] formats
if isinstance(generated, dict):
    variants_list = generated.get("variants", generated.get("prompts", []))
else:
    variants_list = generated

# Wrap each variant in a PromptState
candidates = []
for v in variants_list[:NUM_CANDIDATES]:
    ps = baseline.derive(
        instruction=v["prompt_text"],
        changes_description=v.get("changes_description", v.get("variant_name", "")),
    )
    candidates.append(ps)
    print(f"  {v.get('variant_name', ps.id[:12])}: {v.get('changes_description', '')[:80]}")

print(f"\nGenerated {len(candidates)} candidates (parent: {baseline.id[:12]})")

In [ ]:
#@title Evaluate all candidates
import asyncio

all_eval_results = {}  # prompt_id -> list of result dicts

for idx, candidate in enumerate(candidates):
    label = candidate.changes_description or candidate.id[:12]
    print(f"\n[{idx+1}/{len(candidates)}] Evaluating: {label}")

    candidate_results = []
    _pbar = tqdm(total=len(eval_data), desc=f"Candidate {idx+1}", unit="query")

    for qd in eval_data:
        result = await local_reranker_eval(candidate.render(), qd)
        candidate_results.append(result)
        _pbar.update(1)

    _pbar.close()
    all_eval_results[candidate.id] = candidate_results

    hits = sum(1 for r in candidate_results if r["hit"])
    errors = sum(1 for r in candidate_results if r["error"])
    acc = hits / len(candidate_results) if candidate_results else 0
    avg_c = sum(r["confidence"] for r in candidate_results) / len(candidate_results) if candidate_results else 0

    print(f"  Accuracy: {hits}/{len(candidate_results)} ({acc:.1%})  |  Avg confidence: {avg_c:.3f}  |  Errors: {errors}")

print(f"\nEvaluated {len(candidates)} candidates across {len(eval_data)} queries")
print(f"Total LLM calls: {len(candidates) * len(eval_data)}")

In [ ]:
# --- Compare & select winner ---

import pandas as pd

rows = []

# Baseline row
rows.append({
    "prompt_id": baseline.id[:12],
    "label": "BASELINE (reranker_v1)",
    "hit@1": baseline_hits,
    "total": len(baseline_results),
    "accuracy": baseline_accuracy,
    "avg_confidence": sum(r["confidence"] for r in baseline_results) / len(baseline_results) if baseline_results else 0,
    "errors": sum(1 for r in baseline_results if r["error"]),
})

# Candidate rows
for candidate in candidates:
    cresults = all_eval_results[candidate.id]
    hits = sum(1 for r in cresults if r["hit"])
    total = len(cresults)
    rows.append({
        "prompt_id": candidate.id[:12],
        "label": candidate.changes_description or candidate.id[:12],
        "hit@1": hits,
        "total": total,
        "accuracy": hits / total if total else 0,
        "avg_confidence": sum(r["confidence"] for r in cresults) / total if total else 0,
        "errors": sum(1 for r in cresults if r["error"]),
    })

comparison_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
comparison_df["accuracy"] = comparison_df["accuracy"].map(lambda x: f"{x:.1%}")
comparison_df.index = range(1, len(comparison_df) + 1)
comparison_df.index.name = "rank"

print("FLAT SEARCH RESULTS")
print("=" * 80)
display(comparison_df)

# Select winner
best_row = rows[0]
for r in rows:
    if isinstance(r["accuracy"], float) and r["accuracy"] > (best_row["accuracy"] if isinstance(best_row["accuracy"], float) else 0):
        best_row = r

# Find the PromptState for the winner
best_acc = max(
    (sum(1 for r in all_eval_results[c.id] if r["hit"]) / len(all_eval_results[c.id]), c)
    for c in candidates
)
if best_acc[0] > baseline_accuracy:
    winner = best_acc[1]
    print(f"\nWINNER: {winner.changes_description or winner.id[:12]}")
    print(f"  Accuracy: {best_acc[0]:.1%} (baseline: {baseline_accuracy:.1%}, delta: +{best_acc[0] - baseline_accuracy:.1%})")
    print(f"  PromptState ID: {winner.id}")
    print(f"  Parent ID: {winner.parent_id}")
    print(f"  Lineage: {baseline.id[:12]} → {winner.id[:12]}")
else:
    winner = baseline
    print(f"\nNo candidate beat the baseline ({baseline_accuracy:.1%}). Keeping baseline.")
    print(f"  PromptState ID: {baseline.id}")

# Save winner
store.save_sync(BACKEND_ID, f"optimization/winner_{winner.id[:12]}.json", winner.model_dump())
print(f"\nWinner saved to .promptpotter/projects/{BACKEND_ID}/sync/optimization/")